In [0]:
# from datetime import datetime

# base_path = "abfss://bronze@stweatherprojectnew.dfs.core.windows.net/location_master/raw/"

# files = dbutils.fs.ls(base_path)

# today = datetime.now().strftime("%Y%m%d")

# count = 0

# for file in files:
#     if today in file.name:
#         count += 1

# print(count)

# dbutils.notebook.exit(str(count))

In [0]:
from pyspark.sql.functions import col
import json

status = {}

try:

    # Read Gold Location Master table
    df = spark.table("weather_catalog.gold.location_master")

    # ---------------------------------------------------
    # 1. Row Count
    # ---------------------------------------------------
    row_count = df.count()

    # ---------------------------------------------------
    # 2. Duplicate Key Validation
    # ---------------------------------------------------
    duplicate_count = (
        df.groupBy("Key")
          .count()
          .filter(col("count") > 1)
          .count()
    )

    # ---------------------------------------------------
    # 3. Mandatory Column Null Validation
    # ---------------------------------------------------
    null_count = (
        df.filter(
            col("EnglishName").isNull() |
            col("Key").isNull() |
            col("GeoPosition_Latitude").isNull() |
            col("GeoPosition_Longitude").isNull()
        ).count()
    )

    # ---------------------------------------------------
    # 4. Latitude Validation
    # ---------------------------------------------------
    invalid_latitude = (
        df.filter(
            (col("GeoPosition_Latitude") < -90) |
            (col("GeoPosition_Latitude") > 90)
        ).count()
    )

    # ---------------------------------------------------
    # 5. Longitude Validation
    # ---------------------------------------------------
    invalid_longitude = (
        df.filter(
            (col("GeoPosition_Longitude") < -180) |
            (col("GeoPosition_Longitude") > 180)
        ).count()
    )

    # ---------------------------------------------------
    # Final Validation
    # ---------------------------------------------------
    validation = (
        row_count > 0 and
        duplicate_count == 0 and
        null_count == 0 and
        invalid_latitude == 0 and
        invalid_longitude == 0
    )

    status = {
        "status": "SUCCESS" if validation else "FAIL",
        "row_count": row_count,
        "duplicate_count": duplicate_count,
        "null_count": null_count,
        "invalid_latitude": invalid_latitude,
        "invalid_longitude": invalid_longitude
    }

except Exception as e:

    status = {
        "status": "FAIL",
        "error": str(e)
    }

dbutils.notebook.exit(json.dumps(status))